In [1]:
import pandas as pd
import pandas as pd
from shapely.geometry import Point, LineString
from shapely.wkt import dumps
import numpy as np
from scipy.spatial import cKDTree


def network_update(
    base_node_df: pd.DataFrame,
    base_link_df: pd.DataFrame,
    merge_node_df: pd.DataFrame,
    merge_link_df: pd.DataFrame
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    """
    Integrates and updates node and link IDs from a base network and a network to be merged.

    Args:
        base_node_df (pd.DataFrame): DataFrame of nodes for the base network.
        base_link_df (pd.DataFrame): DataFrame of links for the base network.
        merge_node_df (pd.DataFrame): DataFrame of nodes for the network to be merged.
        merge_link_df (pd.DataFrame): DataFrame of links for the network to be merged.

    Returns:
        tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
            A tuple containing:
            - updated_base_node_df (pd.DataFrame): Updated base node DataFrame.
            - updated_merge_node_df (pd.DataFrame): Updated merged node DataFrame.
            - updated_base_link_df (pd.DataFrame): Updated base link DataFrame.
            - updated_merge_link_df (pd.DataFrame): Updated merged link DataFrame.
    """

    # 1. Integrate and Update Node IDs
    # Save existing node_id to 'old_node_id' column for both base and merge dataframes.
    # If 'old_node_id' already exists, it will be overwritten.
    base_node_df['old_node_id'] = base_node_df['node_id']
    merge_node_df['old_node_id'] = merge_node_df['node_id']

    # Base network's node_ids remain as they are (assuming they start from 1).
    # New node_ids for the merged network start after the maximum node_id of the base network.
    num_rows_in_base_df = len(base_node_df) # Get the total number of rows (nodes) in the base DataFrame.
    max_node_id_in_base_df = base_node_df['node_id'].max() # Get the maximum value in the 'node_id' column.

    # Compare the max_node_id with the number of rows.
    # If they are not equal, the node_ids are not sequential from 1.
    is_base_renumbering_needed = (max_node_id_in_base_df != num_rows_in_base_df)
    if is_base_renumbering_needed:
        print(f"base_node_df: Max ID ({max_node_id_in_base_df}) != Row count ({num_rows_in_base_df}). Renumbering required.")
        # Sort the DataFrame by the existing 'node_id' to maintain a logical order.
        sorted_base_df = base_node_df.sort_values(by='node_id')
        # Reset the index of the sorted DataFrame to ensure it's clean (0, 1, 2, ...).
        reset_base_df = sorted_base_df.reset_index(drop=True)
        # Create a new array of sequential numbers, from 1 to the number of rows.
        new_sequential_ids_for_base = np.arange(1, num_rows_in_base_df + 1)
        
        # Assign the new sequential IDs to the 'node_id' column.
        # We use .copy() to avoid potential pandas SettingWithCopyWarning.
        base_node_df = reset_base_df.copy()
        base_node_df['node_id'] = new_sequential_ids_for_base


    num_rows_in_merge_df = len(merge_node_df) # Get the total number of rows (nodes) in the merge DataFrame.
    max_node_id_in_merge_df = merge_node_df['node_id'].max() # Get the maximum value in the 'node_id' column.

    # Compare the max_node_id with the number of rows.
    is_merge_renumbering_needed = (max_node_id_in_merge_df != num_rows_in_merge_df)
    if is_merge_renumbering_needed:
        print(f"merge_node_df: Max ID ({max_node_id_in_merge_df}) != Row count ({num_rows_in_merge_df}). Renumbering required.")
        # Sort, reset index, and create new sequential IDs.
        sorted_merge_df = merge_node_df.sort_values(by='node_id')
        reset_merge_df = sorted_merge_df.reset_index(drop=True)
        new_sequential_ids_for_merge = np.arange(1, num_rows_in_merge_df + 1)
        
        # Assign the new IDs.
        merge_node_df = reset_merge_df.copy()
        merge_node_df['node_id'] = new_sequential_ids_for_merge

    max_base_node_id = base_node_df['node_id'].max()
    merge_node_df['node_id'] = merge_node_df['node_id'] + max_base_node_id

    # Create a node ID mapping dictionary (old_node_id -> new_node_id)
    # For the base network
    node_id_map = pd.Series(base_node_df['node_id'].values, index=base_node_df['old_node_id']).to_dict()
    # For the merge network
    merge_node_id_map = pd.Series(merge_node_df['node_id'].values, index=merge_node_df['old_node_id']).to_dict()
    

    # 2. Integrate and Update Link IDs
    # Save existing link_id to 'old_link_id' column for both base and merge dataframes.
    # If 'old_link_id' already exists, it will be overwritten.
    base_link_df['old_link_id'] = base_link_df['link_id']
    merge_link_df['old_link_id'] = merge_link_df['link_id']

    num_rows_in_base_link_df = len(base_link_df) # Get the total number of rows (links) in the base DataFrame.
    max_link_id_in_base_link_df = base_link_df['link_id'].max() # Get the maximum value in the 'link_id' column.

    # Compare the max_link_id with the number of rows.
    # If they are not equal, the link_ids are not sequential from 1.
    is_base_link_renumbering_needed = (max_link_id_in_base_link_df != num_rows_in_base_link_df)

    if is_base_link_renumbering_needed:
        print(f"base_link_df: Max ID ({max_link_id_in_base_link_df}) != Row count ({num_rows_in_base_link_df}). Renumbering required.")
        
        # Sort the DataFrame by the existing 'link_id' to maintain a logical order.
        sorted_base_link_df = base_link_df.sort_values(by='link_id')
        
        # Reset the index of the sorted DataFrame to ensure it's clean (0, 1, 2, ...).
        reset_base_link_df = sorted_base_link_df.reset_index(drop=True)
        
        # Create a new array of sequential numbers, from 1 to the number of rows.
        new_sequential_ids_for_base_link = np.arange(1, num_rows_in_base_link_df + 1)
        
        # Assign the new sequential IDs to the 'link_id' column.
        # We use .copy() to avoid potential pandas SettingWithCopyWarning.
        base_link_df = reset_base_link_df.copy()
        base_link_df['link_id'] = new_sequential_ids_for_base_link

    num_rows_in_merge_link_df = len(merge_link_df) # Get the total number of rows (links) in the merge DataFrame.
    max_link_id_in_merge_link_df = merge_link_df['link_id'].max() # Get the maximum value in the 'link_id' column.

    # Compare the max_link_id with the number of rows.
    is_merge_link_renumbering_needed = (max_link_id_in_merge_link_df != num_rows_in_merge_link_df)

    if is_merge_link_renumbering_needed:
        print(f"merge_link_df: Max ID ({max_link_id_in_merge_link_df}) != Row count ({num_rows_in_merge_link_df}). Renumbering required.")
        
        # Sort, reset index, and create new sequential IDs.
        sorted_merge_link_df = merge_link_df.sort_values(by='link_id')
        reset_merge_link_df = sorted_merge_link_df.reset_index(drop=True)
        new_sequential_ids_for_merge_link = np.arange(1, num_rows_in_merge_link_df + 1)
        
        # Assign the new IDs.
        merge_link_df = reset_merge_link_df.copy()
        merge_link_df['link_id'] = new_sequential_ids_for_merge_link

    # New link_ids for the merged network start after the maximum link_id of the base network.
    max_base_link_id = base_link_df['link_id'].max()
    merge_link_df['link_id'] = merge_link_df['link_id'] + max_base_link_id

    # 3. Update from_node_id and to_node_id in Link Files
    # Update base link file using the node ID map
    base_link_df['from_node_id'] = base_link_df['from_node_id'].map(node_id_map)
    base_link_df['to_node_id'] = base_link_df['to_node_id'].map(node_id_map)

    # Update merged link file using the node ID map
    merge_link_df['from_node_id'] = merge_link_df['from_node_id'].map(merge_node_id_map)
    merge_link_df['to_node_id'] = merge_link_df['to_node_id'].map(merge_node_id_map)

    return base_node_df, merge_node_df, base_link_df, merge_link_df



def transfer_connector_builder(
    base_node_df: pd.DataFrame,
    merge_node_df: pd.DataFrame,
    search_radius: float = 1000 # Default search radius in meters
) -> pd.DataFrame:
    """
    Builds connector links between the closest nodes of a base network and a merge network.
    It creates bidirectional links and includes various link attributes.

    Args:
        base_node_df (pd.DataFrame): DataFrame of nodes for the base network.
                                     Must contain 'node_id', 'x_coord', 'y_coord' columns.
        merge_node_df (pd.DataFrame): DataFrame of nodes for the network to be merged.
                                      Must contain 'node_id', 'x_coord', 'y_coord' columns.
        search_radius (float): The maximum distance (in meters) to search for a
                               closest base node for each merge node. Defaults to 1000 meters.

    Returns:
        pd.DataFrame: A DataFrame containing the generated connector links with all specified attributes.
                      Returns an empty DataFrame if no connectors are found or input DataFrames are invalid.
    """

    # Validate input DataFrames
    required_cols = ['node_id', 'x_coord', 'y_coord']
    if not all(col in base_node_df.columns for col in required_cols):
        print(f"Error: base_node_df is missing required columns. Ensure it has {required_cols}")
        return pd.DataFrame()
    if not all(col in merge_node_df.columns for col in required_cols):
        print(f"Error: merge_node_df is missing required columns. Ensure it has {required_cols}")
        return pd.DataFrame()
    
    # Select 'bus_service_node' only in node_type column (added 2026-01)
    # merge_node_df = merge_node_df[merge_node_df['node_type'] == 'bus_service_node']
    # merge_node_df = merge_node_df.reset_index(drop=True)

    # Prepare coordinates for spatial search
    # Assuming x_coord is longitude and y_coord is latitude for haversine distance
    base_coords = base_node_df[['x_coord', 'y_coord']].to_numpy()
    merge_coords = merge_node_df[['x_coord', 'y_coord']].to_numpy()

    # Use cKDTree for efficient nearest neighbor search
    # This will find the closest base node for each merge node
    tree = cKDTree(base_coords)
    # query returns: distances, indices
    # distances are in the same unit as the coordinates, which are degrees
    # We divide by 111320.0 (approx meters per degree at equator) to convert search_radius to degrees.
    # This is a rough approximation for initial candidate filtering; for more accurate distance,
    # Haversine formula is needed *after* finding candidates.
    distances_deg, base_node_indices = tree.query(merge_coords, k=1, distance_upper_bound=search_radius / 111320.0)
    
    connector_links_data = []
    current_link_id = 1
    R = 6371000 # Earth radius in meters

    for i, (merge_node_id, merge_x, merge_y) in merge_node_df[['node_id', 'x_coord', 'y_coord']].iterrows():
        # Check if a candidate base node was found within the approximate radius
        if base_node_indices[i] < len(base_node_df): # Ensure index is valid (not infinity from distance_upper_bound)
            closest_base_node_idx = base_node_indices[i]
            base_node_id = base_node_df.iloc[closest_base_node_idx]['node_id']
            base_x = base_node_df.iloc[closest_base_node_idx]['x_coord']
            base_y = base_node_df.iloc[closest_base_node_idx]['y_coord']

            # Calculate precise Haversine distance in meters
            # Convert degrees to radians
            lat1, lon1, lat2, lon2 = map(np.radians, [merge_y, merge_x, base_y, base_x])
            dlon = lon2 - lon1
            dlat = lat2 - lat1
            a = np.sin(dlat/2)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2)**2
            c = 2 * np.arctan2(np.sqrt(a), np.sqrt(1-a))
            length_meters = R * c

            # Filter by actual search radius in meters
            if length_meters <= search_radius:
                # Create bidirectional connector links

                # Connector: Merge Node -> Base Node
                geometry_wkt_1 = dumps(LineString([(merge_x, merge_y), (base_x, base_y)]))
                connector_links_data.append({
                    "link_id": current_link_id,
                    "name": "",
                    "from_node_id": merge_node_id,
                    "to_node_id": base_node_id,
                    "directed": 1,
                    "geometry": geometry_wkt_1,
                    "dir_flag": 1,
                    "length": length_meters,
                    "lanes": 1,
                    "free_speed": 120,
                    "capacity": 100000,
                    "link_type_name": "connector",
                    "link_type": 0,
                    "allowed_uses": "bike",
                    "from_biway": 1,
                    "is_link": 0,
                    "vdf_toll": 0,
                    "vdf_alpha": 0.15,
                    "vdf_beta": 4,
                    "vdf_plf": 1,
                    "vdf_length_mi": length_meters / 1609.344,
                    "free_speed": 120,
                    "vdf_free_speed_mph": 120/1.609344
                })
                current_link_id += 1

                # Connector: Base Node -> Merge Node (reverse direction)
                geometry_wkt_2 = dumps(LineString([(base_x, base_y), (merge_x, merge_y)]))
                connector_links_data.append({
                    "link_id": current_link_id,
                    "name": "",
                    "from_node_id": base_node_id,
                    "to_node_id": merge_node_id,
                    "directed": 1,
                    "geometry": geometry_wkt_2,
                    "dir_flag": 1,
                    "length": length_meters,
                    "lanes": 1,
                    "free_speed": 120,
                    "capacity": 100000,
                    "link_type_name": "connector",
                    "link_type": 0,
                    "allowed_uses": "bike",
                    "from_biway": 1,
                    "is_link": 0,
                    "vdf_toll": 0,
                    "vdf_alpha": 0.15,
                    "vdf_beta": 4,
                    "vdf_plf": 1,
                    "vdf_length_mi": length_meters / 1609.344,
                    "vdf_free_speed_mph": 120/1.609344
                })
                current_link_id += 1
    
    if not connector_links_data:
        print("No connector links found within the specified search radius.")
        return pd.DataFrame()

    connector_df = pd.DataFrame(connector_links_data)

    # Calculate 'vdf_free_speed_in_mph'
    connector_df['vdf_free_speed_in_mph'] = connector_df['free_speed'] / 1.609344

    # Calculate 'vdf_fftt'
    # Note: free_speed can be 0, which would result in division by zero (infinity)
    connector_df['vdf_fftt'] = (connector_df['length'] / 1000) / connector_df['free_speed'] * 60

    # Handle potential infinite values resulting from division by zero
    # We replace infinity (inf) with 0, a common practice for impassable links
    connector_df.replace([np.inf, -np.inf], 0, inplace=True)


    return connector_df




In [2]:
#import os
#os.chdir("/Users/kong/Library/CloudStorage/OneDrive-TheOhioStateUniversity/CONNECT/green_accessibility_columbus/SourceData")

base_node_df = pd.read_csv('node_updated.csv')
base_link_df = pd.read_csv('link_updated.csv')
merge_node_df = pd.read_csv('bikeway_node_gmns.csv')
merge_link_df = pd.read_csv('bikeway_link_gmns.csv')

updated_base_node_df, updated_merge_node_df, updated_base_link_df, updated_merge_link_df = network_update(
    base_node_df, base_link_df, merge_node_df, merge_link_df
)

updated_base_node_df.to_csv('updated_nodes.csv', index=False)
updated_merge_node_df.to_csv('updated_bikeway_nodes.csv', index=False)
updated_base_link_df.to_csv('updated_links.csv', index=False)
updated_merge_link_df.to_csv('updated_bikeway_links.csv', index=False)

In [3]:
connector_df = transfer_connector_builder(
    base_node_df=updated_base_node_df,
    merge_node_df=updated_merge_node_df,
    search_radius=200
)

connector_df.to_csv('transfer_connector_links.csv', index=False)